# Build Your Own Small GPT — A Guided VS Code Project
**Exercise:** build a GPT-style model from scratch, train it, generate text, and compare attention heads, layers, and embedding sizes.

This notebook is self-contained. No API key, paid service, pretrained model, or dataset download is required. It teaches a character-level, decoder-only transformer, not a reproduction of any proprietary GPT model. Your course's original code was not supplied, so this is an independent educational implementation.

## Start here in VS Code
1. Save this notebook in your `my_GPT` folder and open it in VS Code.
2. Install the Microsoft Python and Jupyter extensions if needed.
3. Click **Select Kernel** and select your project's `venv` Python.
4. Run cells from top to bottom. Begin with `QUICK_RUN = True`.
5. Read each explanation before its code. Once the quick run works, set `QUICK_RUN = False` and run all cells again for longer training.

PyTorch is the only required third-party package. Your CUDA-enabled PyTorch installation can be used directly; this notebook will not replace it. If importing torch fails, install PyTorch into the selected kernel following https://pytorch.org/get-started/locally/. If kernel support is missing, install `ipykernel` in that environment. Restart the kernel after installation.

## The whole project
| Stage | Input | Output |
|---|---|---|
| Tokenizer | Text | Integer character IDs |
| Training examples | IDs | Contexts and next-character targets |
| Embeddings | IDs and positions | Trainable vectors |
| Transformer blocks | Vectors | Context-aware vectors |
| Output layer | Context-aware vectors | Scores for every possible next character |
| Loss and optimizer | Scores and correct targets | Updated model weights |
| Generation | Starting text | Repeated next-character predictions |

The **model predicts**, the **loss measures error**, `backward()` computes gradients, and the **optimizer updates weights**.


## 1. Check Python and choose CPU or GPU
`import` loads a module; `as` gives it a shorter name. `device` specifies where computations happen. Model parameters and input tensors must be on the same device. A small GPU operation tests actual usability, not just detection. We use ordinary float32 arithmetic for this beginner project.


In [ ]:
import sys
import math
import time
import csv
from pathlib import Path
from dataclasses import dataclass, asdict, replace
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        probe = torch.ones(2, 2, device=device)
        _ = probe @ probe
        torch.cuda.synchronize()
        print("GPU:", torch.cuda.get_device_name(0))
        del probe
    except RuntimeError as error:
        print("GPU test failed; using CPU:", error)
        device = torch.device("cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
print("Selected device:", device)

def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


## 2. Choose the training settings
A **hyperparameter** is a setting you choose before training. A **parameter** is a weight learned during training.

| Setting | Meaning |
|---|---|
| `batch_size` | Number of independent text windows per update |
| `context_length` | Maximum characters visible in one window |
| `embedding_size` | Numbers in each token representation |
| `num_heads` | Parallel attention heads per block |
| `num_layers` | Number of transformer blocks |
| `learning_rate` | Optimizer step-size setting |
| `steps` | Number of optimizer updates, not epochs |
| `dropout` | Fraction randomly dropped during training |

We start small for a laptop. `embedding_size` must be divisible by `num_heads`. A dataclass groups related settings; `cfg.embedding_size` accesses one setting. Quick mode checks functionality; it is too short to judge model quality.


In [ ]:
QUICK_RUN = True
RUN_EXPERIMENTS = True
USE_CUSTOM_TEXT = False
TEXT_PATH = Path("input.txt")
OUTPUT_DIR = Path("gpt_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

@dataclass
class GPTConfig:
    vocab_size: int
    context_length: int = 32
    batch_size: int = 8
    embedding_size: int = 64
    num_heads: int = 4
    num_layers: int = 2
    dropout: float = 0.1
    learning_rate: float = 3e-4
    steps: int = 30 if QUICK_RUN else 600
    eval_every: int = 15 if QUICK_RUN else 100
    eval_batches: int = 3 if QUICK_RUN else 10
    seed: int = 42


## 3. Load text
The built-in corpus is an original, deliberately repetitive collection of simple sentences. It lets you learn offline. Its validation results are **practice results, not evidence of broad language ability**: train and validation contain very similar patterns.

For more meaningful experiments, save a varied UTF-8 text corpus as `input.txt` beside this notebook, set `USE_CUSTOM_TEXT = True`, and rerun from here. Prefer tens of thousands of characters or more. The code checks minimum length instead of silently repeating your custom text. `Path.cwd()` shows where relative paths are resolved.


In [ ]:
print("Working directory:", Path.cwd())
subjects = ["the reader", "the teacher", "the child", "the writer", "the artist", "the student"]
actions = ["opens a book", "writes a story", "asks a question", "reads a poem", "shares an idea", "learns a lesson"]
places = ["in the library", "by the window", "in the morning", "after the rain", "near the garden"]
practice_lines = []
for day in range(20):
    for i, subject in enumerate(subjects):
        for j, action in enumerate(actions):
            place = places[(day + i + j) % len(places)]
            practice_lines.append(f"{subject} {action} {place}.\n")
practice_text = "".join(practice_lines)
if USE_CUSTOM_TEXT:
    if not TEXT_PATH.is_file():
        raise FileNotFoundError(f"Create a UTF-8 text file at {TEXT_PATH.resolve()}")
    text = TEXT_PATH.read_text(encoding="utf-8")
else:
    text = practice_text
print("Characters:", len(text))
print(text[:350])


## 4. Build the tokenizer
Each unique character, including a space or newline, gets one ID. IDs are labels, not numerical measures of meaning. `set` removes duplicates, `sorted` gives a consistent order, and `enumerate` supplies an index and item. Dictionaries map characters to IDs and back.

The vocabulary is built from the complete available corpus so validation characters are representable; no validation sequences are used for gradient updates. Unseen prompt characters raise a helpful error.


In [ ]:
characters = sorted(set(text))
char_to_id = {character: index for index, character in enumerate(characters)}
id_to_char = {index: character for index, character in enumerate(characters)}
vocab_size = len(characters)

def encode(string):
    unknown = sorted(set(string) - set(char_to_id))
    if unknown:
        raise ValueError(f"Characters outside vocabulary: {unknown!r}. Use characters from the corpus.")
    return [char_to_id[character] for character in string]

def decode(ids):
    return "".join(id_to_char[int(index)] for index in ids)

cfg = GPTConfig(vocab_size=vocab_size)
example = text[:12]
print("Vocabulary size:", vocab_size)
print("Text:", repr(example))
print("IDs:", encode(example))
print("Decoded:", repr(decode(encode(example))))
assert decode(encode(example)) == example


## 5. Split data and create next-character examples
We split the token stream **before** sampling windows, so no window crosses the train/validation boundary. Validation measures predictions without updating weights.

For `hello`, input `hell` has target `ello`. At every position the correct answer is the following character. Python slices exclude their ending index.

A batch has shape `[B, T]`: `B` examples and `T` characters per example. `torch.stack` stacks equal-length windows into a batch. The dataset stays on CPU; only each batch moves to the chosen device.


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
split_index = int(0.9 * len(data))
train_data = data[:split_index]
val_data = data[split_index:]
for name, source in [("train", train_data), ("validation", val_data)]:
    if len(source) <= cfg.context_length:
        raise ValueError(f"Too little {name} text. Add more text or reduce context_length.")
print("Train / validation tokens:", len(train_data), len(val_data))

def get_batch(split, config, generator=None):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - config.context_length,
                           (config.batch_size,), generator=generator)
    x = torch.stack([source[i:i + config.context_length] for i in starts.tolist()])
    y = torch.stack([source[i + 1:i + config.context_length + 1] for i in starts.tolist()])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", cfg)
print("Input shape:", xb.shape, "Target shape:", yb.shape)
print("Input:", repr(decode(xb[0].tolist())))
print("Target:", repr(decode(yb[0].tolist())))
for position in range(4):
    print(repr(decode(xb[0, :position + 1].tolist())), "->", repr(decode([yb[0, position].item()])))


## 6. Understand embeddings before building the model
`nn.Embedding(V, C)` creates a table with `V` rows and `C` trainable numbers per row. Calling it with IDs retrieves vectors. Token embeddings describe identity; position embeddings represent locations. Adding them preserves the vector width.

Shape notation used below: **B** = batch size, **T** = sequence length, **C** = embedding size, **H** = heads, **D** = C/H, **V** = vocabulary size.

These demonstration layers are separate from the actual model we will construct later.


In [ ]:
token_table = nn.Embedding(vocab_size, cfg.embedding_size).to(device)
position_table = nn.Embedding(cfg.context_length, cfg.embedding_size).to(device)
positions = torch.arange(xb.shape[1], device=device)
token_vectors = token_table(xb)               # [B, T, C]
position_vectors = position_table(positions)  # [T, C]
hidden = token_vectors + position_vectors    # Positions broadcast across B
print("Token vectors:", token_vectors.shape)
print("Position vectors:", position_vectors.shape)
print("Combined vectors:", hidden.shape)
del token_table, position_table, token_vectors, position_vectors, hidden


## 7. Build causal multi-head attention
A Python **class** defines a reusable component. `nn.Module` is PyTorch's base class for trainable components. `__init__` creates its layers; `self` means this instance; `super().__init__()` initializes PyTorch's bookkeeping. `forward` describes the computation. Calling `layer(x)` runs its `forward` method.

Attention creates three learned projections:
- **Q (query):** used to score what this position attends to.
- **K (key):** compared with queries to compute those scores.
- **V (value):** information mixed using the resulting weights.

We compute `Q @ K.transpose(...) / sqrt(D)`, mask future positions, apply softmax, and multiply by V. `@` means matrix multiplication. Softmax turns each row of scores into weights summing to one (before dropout). These are attention weights, not probabilities of next characters.

`reshape` splits C into H heads of width D. `transpose` moves the head dimension so each head computes its own attention. After attention, we join the heads and apply an output projection. The causal mask allows position t to see positions 0 through t, never later positions.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        if config.embedding_size % config.num_heads != 0:
            raise ValueError("embedding_size must be divisible by num_heads")
        self.num_heads = config.num_heads
        self.head_size = config.embedding_size // config.num_heads
        self.qkv = nn.Linear(config.embedding_size, 3 * config.embedding_size)
        self.projection = nn.Linear(config.embedding_size, config.embedding_size)
        self.attention_dropout = nn.Dropout(config.dropout)
        self.output_dropout = nn.Dropout(config.dropout)
        # A buffer moves with the model but is not optimized as a parameter.
        self.register_buffer("causal_mask", torch.tril(torch.ones(
            config.context_length, config.context_length, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.reshape(B, T, self.num_heads, self.head_size).transpose(1, 2)
        k = k.reshape(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v.reshape(B, T, self.num_heads, self.head_size).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attention_dropout(weights)
        mixed = weights @ v                     # [B, H, T, D]
        mixed = mixed.transpose(1, 2).contiguous().reshape(B, T, C)
        return self.output_dropout(self.projection(mixed))

print("Allowed positions (1 = visible):")
print(torch.tril(torch.ones(4, 4, dtype=torch.int64)))


## 8. Build one transformer block
Each block contains attention and a feed-forward network. Attention mixes information **between positions**; the feed-forward network transforms features **at each position** with shared weights.

`LayerNorm(C)` normalizes each token's C features. `Linear(C, 4*C)` expands the representation; GELU provides a nonlinear transformation; a second linear layer returns to C features. `nn.Sequential` runs its layers in order.

The `x + ...` expressions are **residual connections**. They retain the incoming representation while adding a learned update. This is a pre-normalization block: normalization happens before each sublayer.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config.embedding_size
        self.norm1 = nn.LayerNorm(C)
        self.attention = CausalSelfAttention(config)
        self.norm2 = nn.LayerNorm(C)
        self.feed_forward = nn.Sequential(
            nn.Linear(C, 4 * C),
            nn.GELU(),
            nn.Linear(4 * C, C),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x


## 9. Assemble GPT and calculate loss
The full model adds embeddings, passes vectors through the blocks, normalizes them, and produces vocabulary-sized scores called **logits**.

Input shape: `[B,T]`. Logits shape: `[B,T,V]`. We flatten the first two dimensions to compare every position's prediction with its target. Cross-entropy expects raw logits here; **do not apply softmax before this loss**.

`targets=None` lets us use the same model without answers during generation. `loss=None` means no loss was requested, not zero prediction error. `ModuleList` registers all blocks so their parameters are included in optimization and saving.


In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.embedding_size)
        self.position_embedding = nn.Embedding(config.context_length, config.embedding_size)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.num_layers)])
        self.final_norm = nn.LayerNorm(config.embedding_size)
        self.output_head = nn.Linear(config.embedding_size, config.vocab_size)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        if T > self.config.context_length:
            raise ValueError("Input exceeds context_length")
        positions = torch.arange(T, device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        logits = self.output_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(B * T, self.config.vocab_size),
                                   targets.reshape(B * T))
        return logits, loss

set_seed(cfg.seed)
model = MiniGPT(cfg).to(device)
logits, loss = model(xb, yb)
print("Logits:", logits.shape)
print("Initial loss:", loss.item())
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


## 10. Check that the model cannot see the future
Change the second half of an input. Predictions in the unchanged first half should stay the same. `eval()` turns off dropout so random dropping does not interfere with this check. `no_grad()` avoids recording gradient computations. This is a meaningful check of the causal mask.


In [ ]:
model.eval()
with torch.no_grad():
    original = xb[:1].clone()
    changed = original.clone()
    midpoint = cfg.context_length // 2
    changed[:, midpoint:] = (changed[:, midpoint:] + 1) % vocab_size
    before, _ = model(original)
    after, _ = model(changed)
    assert torch.allclose(before[:, :midpoint], after[:, :midpoint], atol=1e-5, rtol=1e-5)
model.train()
print("Causal mask check passed.")


## 11. Train the model
One training update has five key operations:
1. Get contexts and targets.
2. Compute logits and loss.
3. `zero_grad()` clears old gradients.
4. `loss.backward()` computes how weights affect the loss.
5. `optimizer.step()` changes weights using those gradients.

Gradient clipping limits the combined gradient norm to help control unusually large updates. AdamW is the optimizer. `model.train()` enables dropout; `model.eval()` disables it. Evaluation uses no weight updates.

Evaluation windows are sampled with a fixed, separate generator so repeated evaluations and architecture comparisons use the same windows without consuming the training batch generator. Lower validation loss is generally better. A falling training loss with rising validation loss suggests overfitting.

We retain the best validation checkpoint, including the initial model if later steps do not improve it. The training loop prints progress; an update is not one full pass over the corpus.


In [ ]:
@torch.no_grad()
def evaluate(model, config):
    was_training = model.training
    model.eval()
    result = {}
    for split in ("train", "val"):
        rng = torch.Generator().manual_seed(2026 if split == "train" else 2027)
        losses = []
        for _ in range(config.eval_batches):
            x, y = get_batch(split, config, generator=rng)
            _, batch_loss = model(x, y)
            losses.append(batch_loss.item())
        result[split] = sum(losses) / len(losses)
    model.train(was_training)
    return result

def train_model(config, verbose=True):
    set_seed(config.seed)
    net = MiniGPT(config).to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=config.learning_rate)
    batch_rng = torch.Generator().manual_seed(config.seed)
    history = []
    best_loss = float("inf")
    best_state = None
    if device.type == "cuda":
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    for step in range(config.steps + 1):
        if step % config.eval_every == 0 or step == config.steps:
            metrics = evaluate(net, config)
            history.append({"step": step, "train_loss": metrics["train"], "val_loss": metrics["val"]})
            if metrics["val"] < best_loss:
                best_loss = metrics["val"]
                best_state = {key: value.detach().cpu().clone() for key, value in net.state_dict().items()}
            if verbose:
                print(f"step {step:4d} | train {metrics['train']:.3f} | val {metrics['val']:.3f}")
        if step == config.steps:
            break
        net.train()
        x, y = get_batch("train", config, generator=batch_rng)
        _, loss = net(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time
    net.load_state_dict(best_state)
    net.eval()
    return net, history, elapsed

# Replace the untrained demonstration model with a freshly trained model.
del model
model, history, training_seconds = train_model(cfg)
print(f"Training and evaluations took {training_seconds:.1f} seconds.")
print("Restored the checkpoint with lowest measured validation loss.")


## 12. Inspect the learning curve
The table always works. A plot appears if matplotlib is installed; plotting is optional. Perplexity is `exp(cross_entropy)` and should only be compared for the same tokenization and evaluation data. Our quick run and repetitive practice text do not support strong claims about language quality.


In [ ]:
for row in history:
    print(f"{row['step']:4d}  train={row['train_loss']:.3f}  val={row['val_loss']:.3f}")
print("Best validation perplexity:", round(math.exp(min(r["val_loss"] for r in history)), 2))
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Optional plot skipped. To enable it, install matplotlib in this kernel.")
else:
    plt.figure(figsize=(7, 4))
    plt.plot([r["step"] for r in history], [r["train_loss"] for r in history], label="Train")
    plt.plot([r["step"] for r in history], [r["val_loss"] for r in history], label="Validation")
    plt.xlabel("Optimizer updates")
    plt.ylabel("Cross-entropy loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 13. Generate text
Generation repeatedly feeds the most recent context into the model and uses the **last position's logits** to choose the next character. `softmax` now converts vocabulary logits into probabilities; `multinomial` samples an ID; `cat` appends it.

Temperature below 1 sharpens the distribution; above 1 makes it flatter. It does not improve the learned model. When the text becomes longer than the context window, we keep its most recent characters as input. This simple model uses positions starting at zero within each window.

A quick run may produce nonsense. Longer training on the tiny corpus may reproduce its sentences; that is not general conversation ability.


In [ ]:
@torch.no_grad()
def generate(net, prompt, new_tokens=200, temperature=0.8):
    if not prompt:
        raise ValueError("Use a nonempty prompt.")
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    was_training = net.training
    net.eval()
    model_device = next(net.parameters()).device
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=model_device)
    for _ in range(new_tokens):
        context = ids[:, -net.config.context_length:]
        logits, _ = net(context)
        probabilities = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1)
        ids = torch.cat((ids, next_id), dim=1)
    net.train(was_training)
    return decode(ids[0].tolist())

prompt = text[:12]  # Works with either built-in or custom vocabulary.
set_seed(123)
sample = generate(model, prompt, new_tokens=120 if QUICK_RUN else 300)
print(sample)


## 14. Complete the hyperparameter exercise
Each experiment trains a **new model from scratch**. We hold the data, context length, batch size, learning rate, update count, and sampling seeds fixed. Each change is relative to the baseline:

| Experiment | Embedding size | Heads | Layers | Question |
|---|---:|---:|---:|---|
| Baseline | 64 | 4 | 2 | Reference |
| Fewer heads | 64 | 2 | 2 | Effect of head partitioning? |
| More heads | 64 | 8 | 2 | Effect of more, narrower heads? |
| More layers | 64 | 4 | 3 | Effect of depth? |
| Wider embeddings | 128 | 4 | 2 | Effect of width? |

At fixed embedding size, changing heads in this implementation keeps the learned parameter count unchanged, but changes each head's width. Increasing layers or embedding size increases parameter count and compute. Equal updates do not mean equal compute budgets.

Quick mode tests the experiment pipeline; use longer runs and several seeds for meaningful comparisons. Validation selects checkpoints and is used for these comparisons, so it is not an untouched final test set. For a serious report add a separate test split. Set `RUN_EXPERIMENTS = False` to skip additional training.


In [ ]:
results = []
def result_row(name, config, net, hist, seconds):
    best = min(hist, key=lambda row: row["val_loss"])
    return {"experiment": name, "embedding_size": config.embedding_size,
            "heads": config.num_heads, "layers": config.num_layers,
            "parameters": sum(p.numel() for p in net.parameters()),
            "best_step": best["step"], "train_loss": best["train_loss"],
            "val_loss": best["val_loss"], "seconds": seconds}

results.append(result_row("baseline", cfg, model, history, training_seconds))
if RUN_EXPERIMENTS:
    variants = [
        ("fewer_heads", replace(cfg, num_heads=2)),
        ("more_heads", replace(cfg, num_heads=8)),
        ("more_layers", replace(cfg, num_layers=3)),
        ("wider_embeddings", replace(cfg, embedding_size=128)),
    ]
    for name, experiment_cfg in variants:
        print("\nExperiment:", name)
        experiment_model, experiment_history, seconds = train_model(experiment_cfg)
        results.append(result_row(name, experiment_cfg, experiment_model, experiment_history, seconds))
        set_seed(123)
        print(generate(experiment_model, prompt, new_tokens=80))
        del experiment_model
        if device.type == "cuda":
            torch.cuda.empty_cache()

print("\nExperiment          C   H   L   Parameters    Val loss   Seconds")
for row in results:
    print(f"{row['experiment']:19s} {row['embedding_size']:3d} {row['heads']:3d} {row['layers']:3d} "
          f"{row['parameters']:10d} {row['val_loss']:11.4f} {row['seconds']:9.1f}")
with (OUTPUT_DIR / "experiments.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)


## 15. Save and reload your baseline model
A `state_dict` contains learned tensors. To reconstruct the model we also save its configuration and exact vocabulary. This checkpoint supports inference; it does not save optimizer state for exact training resumption.

The reload check compares logits, not random generated samples. Load only checkpoints you trust. The next cell saves and reloads your own checkpoint with `weights_only=True`.

Output files are written to the `gpt_outputs` folder. Rerunning these cells replaces files with these same names.


In [ ]:
checkpoint_path = OUTPUT_DIR / "mini_gpt.pt"
torch.save({
    "config": asdict(cfg),
    "characters": characters,
    "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},
}, checkpoint_path)
(OUTPUT_DIR / "sample.txt").write_text(sample, encoding="utf-8")

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
loaded_cfg = GPTConfig(**checkpoint["config"])
assert checkpoint["characters"] == characters
loaded_model = MiniGPT(loaded_cfg).to(device)
loaded_model.load_state_dict(checkpoint["model_state"])
loaded_model.eval()
model.eval()
with torch.no_grad():
    expected, _ = model(xb)
    actual, _ = loaded_model(xb)
    assert torch.allclose(expected, actual, atol=1e-5, rtol=1e-5)
print("Reload check passed.")
print("Files saved to:", OUTPUT_DIR.resolve())
print(generate(loaded_model, prompt, new_tokens=80))


## 16. Your exercise report
After running longer experiments, complete these notes using your actual results:

1. **Data:** built-in practice corpus or your own text? How many characters and unique tokens?
2. **Baseline:** embedding size, heads, layers, parameters, updates, train loss, validation loss.
3. **Heads:** compare 2, 4, and 8. Did validation loss change? Why does parameter count stay constant here?
4. **Layers:** compare 2 and 3. Was any gain worth the training time?
5. **Embedding size:** compare 64 and 128. How did parameter count and validation loss change?
6. **Generated examples:** keep the same prompt and temperature for comparisons.
7. **Limitations:** small/repetitive data, short runs, one seed, validation reused for selection, no independent test evaluation.

Do not assume bigger is always better. Repeat promising comparisons with different `seed` values. Preserve `experiments.csv` before rerunning if you want to retain earlier results.

## Troubleshooting
| Symptom | Action |
|---|---|
| `No module named torch` | Select the right kernel or install PyTorch into that environment |
| NumPy warning | Run `%pip install numpy` in a separate cell, restart kernel, rerun |
| CPU selected | Check `torch.__version__` and CUDA availability in this exact kernel |
| CUDA out of memory | Reduce batch size, context length, or embedding size; restart kernel and rerun |
| CPU/GPU device mismatch | Restart and run all cells in order; avoid moving only part of the model |
| `NameError` | A previous cell was skipped or the kernel restarted |
| Unknown prompt characters | Use characters present in the training vocabulary |
| Text file not found | Check printed working directory and `TEXT_PATH` |
| Gibberish | Quick mode is a functional test; train longer and use a richer corpus |
| Slow execution | Start with quick mode and disable experiments while debugging |

## Read the code again in this order
`get_batch` → `MiniGPT.forward` → `TransformerBlock.forward` → `CausalSelfAttention.forward` → `train_model` → `generate`.

You now have a complete pipeline: text becomes IDs; embeddings become context-aware features; logits are compared with next-character answers; training updates weights; generation repeatedly samples the next character.

## Official references
- [PyTorch Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding)
- [PyTorch Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- [PyTorch LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.normalization.LayerNorm.html)
- [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [PyTorch AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW)

This notebook implements attention explicitly to make its operations visible rather than calling a pretrained model or an attention API that hides the steps.
